# Feature extraction - ekstrakcja cech


To see if virtual enviroment works properly, run this two commands below:

In [8]:
# import sys
# print(sys.executable)

### Import Python libraries

In [9]:
import numpy as np
import pandas as pd
import sklearn

sklearn.__version__

'1.8.0'

### Load data

In [14]:
def fetch_financial_data(company='AMZN'):
    """
    Function to fetch stock market quotations.
    """
    import pandas_datareader.data as web
    return web.DataReader(name=company, data_source='stooq')

try:
    df_raw = fetch_financial_data()
except TypeError:
    # Fallback for pandas_datareader/pandas compatibility issue
    df_raw = (
        pd.read_csv(
            "https://stooq.com/q/d/l/?s=amzn.us&i=d",
            parse_dates=["Date"],
            index_col="Date",
        )
        .sort_index(ascending=False)
    )
df_raw.head()

,Open,High,Low,Close,Volume
Date,,,,,
2026-03-17,212.780,215.700,212.430,215.20,30722286
2026-03-16,208.350,212.724,207.445,211.74,42209316
2026-03-13,209.605,210.560,206.220,207.67,35662137
2026-03-12,210.390,211.710,208.150,209.53,44349501
2026-03-11,215.705,217.000,211.350,212.65,34199303


### Copy dataset


In [15]:
df = df_raw.copy()
df = df[:5]
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 5 entries, 2026-03-17 to 2026-03-11
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Open    5 non-null      float64
 1   High    5 non-null      float64
 2   Low     5 non-null      float64
 3   Close   5 non-null      float64
 4   Volume  5 non-null      int64  
dtypes: float64(4), int64(1)
memory usage: 240.0 bytes


### New variables

In [19]:
print(df.index.day) # extract day from datetime index
df['day'] = df.index.day
df['month'] = df.index.month
df['year'] = df.index.year
df

Index([17, 16, 13, 12, 11], dtype='int32', name='Date')


,Open,High,Low,Close,Volume,day,month,year
Date,,,,,,,,
2026-03-17,212.780,215.700,212.430,215.20,30722286,17,3,2026
2026-03-16,208.350,212.724,207.445,211.74,42209316,16,3,2026
2026-03-13,209.605,210.560,206.220,207.67,35662137,13,3,2026
2026-03-12,210.390,211.710,208.150,209.53,44349501,12,3,2026
2026-03-11,215.705,217.000,211.350,212.65,34199303,11,3,2026


### Dyscretization of a continuous variable - dyskretyzacja zmiennej ciągłej

In [21]:
df = pd.DataFrame(data={'height': [175., 178.5,  185., 191., 184.5, 183., 168.]})
df

,height
0,175.0
1,178.5
2,185.0
3,191.0
4,184.5
5,183.0
6,168.0


In [23]:
df['height_cat'] = pd.cut(x=df.height, bins=3) # cut into 3 equal-width bins
df

,height,height_cat
0,175.0,"(167.977, 175.667]"
1,178.5,"(175.667, 183.333]"
2,185.0,"(183.333, 191.0]"
3,191.0,"(183.333, 191.0]"
4,184.5,"(183.333, 191.0]"
5,183.0,"(175.667, 183.333]"
6,168.0,"(167.977, 175.667]"


In [25]:
df['height_cat'] = pd.cut(x=df.height, bins=(160, 175, 180, 195)) # cut into custom bins as tuples
df

,height,height_cat
0,175.0,"(160, 175]"
1,178.5,"(175, 180]"
2,185.0,"(180, 195]"
3,191.0,"(180, 195]"
4,184.5,"(180, 195]"
5,183.0,"(180, 195]"
6,168.0,"(160, 175]"


In [27]:
df['height_cat'] = pd.cut(x=df.height, bins=(160, 175, 180, 195), labels=['small', 'medium', 'high']) # cut into custom bins with labels
df

,height,height_cat
0,175.0,small
1,178.5,medium
2,185.0,high
3,191.0,high
4,184.5,high
5,183.0,high
6,168.0,small


In [28]:
pd.get_dummies(df, drop_first=True, prefix='height')

,height,height_medium,height_high
0,175.0,False,False
1,178.5,True,False
2,185.0,False,True
3,191.0,False,True
4,184.5,False,True
5,183.0,False,True
6,168.0,False,False


In [29]:
df = pd.DataFrame(data={'lang': [['PL', 'ENG'], ['GER', 'ENG', 'PL', 'FRA'], ['RUS']]})
df

,lang
0,"[PL, ENG]"
1,"[GER, ENG, PL, FRA]"
2,[RUS]


In [30]:
df['lang_number'] = df['lang'].apply(len) # extract number of languages spoken
df

,lang,lang_number
0,"[PL, ENG]",2
1,"[GER, ENG, PL, FRA]",4
2,[RUS],1


In [32]:
df['PL_flag'] = df['lang'].apply(lambda x: 1 if 'PL' in x else 0) # extract flag for Polish language, if yes 1 if no 0
df

,lang,lang_number,PL_flag
0,"[PL, ENG]",2,1
1,"[GER, ENG, PL, FRA]",4,1
2,[RUS],1,0


In [33]:
df = pd.DataFrame(data={'website': ['wp.pl', 'onet.pl', 'google.com']})
df

,website
0,wp.pl
1,onet.pl
2,google.com


In [34]:
df.website.str.split(',', expand=True) # split website into parts by comma, expand into separate columns

,0
0,wp.pl
1,onet.pl
2,google.com


In [35]:
new = df.website.str.split('.', expand=True)
df['portal'] = new[0]
df['extension'] = new[1]
df

,website,portal,extension
0,wp.pl,wp,pl
1,onet.pl,onet,pl
2,google.com,google,com


## Summary

### Goal
This notebook demonstrates **feature extraction (ekstrakcja cech)** on different data types:
- time-series stock data,
- continuous numeric data,
- list/categorical text data,
- string parsing from URLs/domains.

### Continuous variable discretization - dyskretyzacja ciągłych zmiennych
- Replaced `df` with a height dataset.
- Created categorical bins with:
    - equal-width bins (`bins=3`)
    - custom ranges (`(160, 175, 180, 195)`)
    - labeled bins (`small`, `medium`, `high`)
- Applied one-hot encoding with `pd.get_dummies(..., drop_first=True)`.

### Methods used in this notebook

- `fetch_financial_data(company='AMZN')` + `web.DataReader(...)`  
    Downloads historical stock data (OHLCV) from **Stooq** for a selected ticker.

- `try/except` fallback with `pd.read_csv(...)`  
    If API download fails, data is loaded directly from a CSV URL.

- `df_raw.copy()` and row slicing (`df = df[:5]`)  
    Creates an independent working copy and limits rows for quick demonstrations.

- `df.index.day / month / year`  
    Extracts calendar features from a `DatetimeIndex` (feature extraction from time data).

- `pd.cut(...)`  
    Discretizes continuous values (height) into bins:
    - equal-width bins (`bins=3`)
    - custom bin edges (`bins=(160, 175, 180, 195)`)
    - custom labels (`small`, `medium`, `high`)

- `pd.get_dummies(..., drop_first=True)`  
    One-hot encodes categorical columns into numeric indicator variables.

- `Series.apply(len)`  
    Computes derived numeric feature from list data (number of languages).

- `Series.apply(lambda ...)`  
    Builds a binary flag feature (e.g., whether `'PL'` appears in language list).

- `Series.str.split(..., expand=True)`  
    Splits string features into structured columns (e.g., `portal` and `extension`).